## 🎯 Learning Objectives
* Understand the motivation and mechanisms behind mixed precision training for LLMs.
* Implement automatic mixed precision (AMP) using PyTorch's `torch.cuda.amp`.
* Grasp the concept and application of gradient checkpointing to reduce memory footprint during training.
* Apply gradient checkpointing using PyTorch's `torch.utils.checkpoint`.
* Analyze the performance trade-offs and synergistic benefits of combining mixed precision and gradient checkpointing in large-scale LLM training.


## FT02-L06: Mixed Precision Training and Gradient Checkpointing

Training colossal Large Language Models (LLMs) like GPT-4 or beyond demands an immense amount of computational resources, particularly GPU memory. As models scale in parameters and context length, the memory required to store activations, gradients, and optimizer states can quickly exceed even the most powerful accelerators. This lesson explores two critical techniques—**Mixed Precision Training** and **Gradient Checkpointing**—that are indispensable for pushing the boundaries of LLM scale and efficiency.

### Mixed Precision Training: The Efficiency Boost

Imagine you're an architect designing a skyscraper. You need extremely precise measurements for the foundation and structural integrity (e.g., down to millimeters), but for interior finishes or preliminary sketches, a less precise measurement (e.g., centimeters) might be perfectly adequate and much faster to work with. Mixed precision training applies a similar logic to neural network computations.

Traditionally, deep learning models use 32-bit floating-point numbers (FP32 or `float32`) for all calculations. While highly precise, FP32 consumes significant memory and computational bandwidth. Modern GPUs, especially NVIDIA's Tensor Cores, are highly optimized for 16-bit floating-point numbers (FP16 or `half`).

**How it works:**

1.  **FP16 for most operations:** The bulk of computations, particularly matrix multiplications and convolutions, are performed using FP16. This halves the memory footprint for activations and gradients and significantly speeds up calculations on compatible hardware.
2.  **FP32 for critical operations:** Certain operations, like weight updates, loss calculations, and some accumulation steps, are still performed in FP32 to maintain numerical stability and prevent issues like *underflow* (numbers becoming too small to represent) or *overflow* (numbers becoming too large).
3.  **Dynamic Loss Scaling:** A crucial component of mixed precision. When using FP16, small gradients can underflow to zero, leading to vanishing gradients. Loss scaling multiplies the loss by a large scalar before computing gradients. This pushes the FP16 gradients into a representable range. After computing gradients, they are unscaled before the FP32 weight update.

**Benefits:**
*   **Reduced Memory Footprint:** Halves the memory for activations and gradients, allowing larger models or batch sizes.
*   **Faster Training:** Leverages specialized hardware (Tensor Cores) for FP16 operations, leading to significant speedups (often 2-3x).
*   **Energy Efficiency:** Less data movement and faster computation can lead to lower power consumption.

### Gradient Checkpointing: The Memory Saver

During the backward pass (backpropagation), the neural network needs access to the intermediate activation values computed during the forward pass. For very deep networks or large models, storing all these activations in memory can be prohibitive. Gradient checkpointing offers an elegant solution by trading computation for memory.

Think of it like reading a long book. Instead of keeping every page open and highlighted to remember details for a quiz at the end, you only bookmark a few key pages. If you need details from an un-bookmarked page, you quickly re-read that section. It takes a little more time to re-read, but you don't need to keep the whole book open.

**How it works:**

1.  **Selective Storage:** Instead of storing all intermediate activations from the forward pass, only a subset of activations (at 


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
import os

# --- Configuration --- 
# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Define a Simple LLM-like Block --- 
# We'll simulate a deep, memory-intensive block, like a Transformer layer
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(), # Modern activation function
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Self-attention block
        attn_output, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + self.dropout(attn_output)
        
        # Feed-forward block
        ffn_output = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_output)
        return x

class SimpleLLM(nn.Module):
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)
        ])
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.token_embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.output_layer(x)

# --- Model Parameters (scaled down for demonstration) ---
VOCAB_SIZE = 10000
EMBED_DIM = 512 # Typical embedding dimension
NUM_HEADS = 8
FF_DIM = 2048 # Feed-forward dimension
NUM_LAYERS = 12 # A moderately deep model
BATCH_SIZE = 4
SEQ_LEN = 128 # Sequence length

# --- Instantiate Model and Optimizer ---
model = SimpleLLM(NUM_LAYERS, EMBED_DIM, NUM_HEADS, FF_DIM, VOCAB_SIZE).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# --- Dummy Data --- 
# Input tokens (batch_size, sequence_length)
input_data = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
# Target tokens (for simplicity, next token prediction)
target_data = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)

# --- Loss Function --- 
criterion = nn.CrossEntropyLoss(ignore_index=-1) # -1 for padding if applicable

print(f"Model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

# --- 2. Mixed Precision Training (AMP) Demonstration --- 
print("\n--- Demonstrating Mixed Precision Training (AMP) ---")
scaler = GradScaler()

# Training step with AMP
optimizer.zero_grad()
with autocast(): # Enables mixed precision for operations within this context
    output_amp = model(input_data)
    # Reshape for CrossEntropyLoss: (N, C, d1, d2...) for input, (N, d1, d2...) for target
    loss_amp = criterion(output_amp.view(-1, VOCAB_SIZE), target_data.view(-1))

# Scales loss appropriately for FP16 to prevent underflow
scaler.scale(loss_amp).backward()

# Unscales gradients and calls optimizer.step()
# If gradients are NaN or Inf, it skips the step and adjusts the scale
scaler.step(optimizer)
scaler.update() # Updates the scale factor for the next iteration

print(f"AMP Loss: {loss_amp.item():.4f}")
print("Mixed precision training step completed. Gradients were scaled and unscaled automatically.")

# --- 3. Gradient Checkpointing Demonstration --- 
print("\n--- Demonstrating Gradient Checkpointing ---")

# Reset model and optimizer for a clean checkpointing demo
model_checkpoint = SimpleLLM(NUM_LAYERS, EMBED_DIM, NUM_HEADS, FF_DIM, VOCAB_SIZE).to(device)
optimizer_checkpoint = optim.AdamW(model_checkpoint.parameters(), lr=1e-4)

# Training step with Gradient Checkpointing
optimizer_checkpoint.zero_grad()

# We'll apply checkpointing to a subset of layers to illustrate
# In a real LLM, you'd checkpoint entire transformer blocks or groups of blocks.

def forward_with_checkpoint(model_input):
    x = model_checkpoint.token_embedding(model_input)
    for i, layer in enumerate(model_checkpoint.layers):
        # Checkpoint every other layer, or a specific range
        if i % 2 == 0: # Checkpoint even-indexed layers
            x = checkpoint(layer, x, use_reentrant=False) # use_reentrant=False is recommended for PyTorch 1.11+
        else:
            x = layer(x)
    return model_checkpoint.output_layer(x)

output_checkpoint = forward_with_checkpoint(input_data)
loss_checkpoint = criterion(output_checkpoint.view(-1, VOCAB_SIZE), target_data.view(-1))

loss_checkpoint.backward()
optimizer_checkpoint.step()

print(f"Checkpointing Loss: {loss_checkpoint.item():.4f}")
print("Gradient checkpointing step completed. Activations for checkpointed layers were recomputed during backward pass.")

# --- 4. Combining AMP and Gradient Checkpointing --- 
print("\n--- Demonstrating Combined AMP and Gradient Checkpointing ---")

# Reset model and optimizer
model_combined = SimpleLLM(NUM_LAYERS, EMBED_DIM, NUM_HEADS, FF_DIM, VOCAB_SIZE).to(device)
optimizer_combined = optim.AdamW(model_combined.parameters(), lr=1e-4)
scaler_combined = GradScaler()

optimizer_combined.zero_grad()

with autocast():
    x_combined = model_combined.token_embedding(input_data)
    for i, layer in enumerate(model_combined.layers):
        if i % 2 == 0: # Checkpoint even-indexed layers
            # Note: When combining AMP and checkpointing, ensure the checkpointed function
            # (the 'layer' in this case) is called within the autocast context.
            # The checkpoint function itself handles the recomputation in FP32 if needed,
            # but the initial forward pass within autocast will use FP16 where possible.
            x_combined = checkpoint(layer, x_combined, use_reentrant=False)
        else:
            x_combined = layer(x_combined)
    output_combined = model_combined.output_layer(x_combined)
    loss_combined = criterion(output_combined.view(-1, VOCAB_SIZE), target_data.view(-1))

scaler_combined.scale(loss_combined).backward()
scaler_combined.step(optimizer_combined)
scaler_combined.update()

print(f"Combined Loss: {loss_combined.item():.4f}")
print("Combined AMP and Gradient Checkpointing step completed.")

# --- Memory Usage (Conceptual) --- 
# Actual memory usage is highly dependent on GPU, model size, and PyTorch version.
# To observe real memory savings, you would typically run this on a GPU with a much larger model
# and use tools like `nvidia-smi` or `torch.cuda.max_memory_allocated()`.
# For this simple example, we just demonstrate the API usage.

# Example of how to check max memory allocated (requires GPU)
if torch.cuda.is_available():
    print(f"\nMax memory allocated by PyTorch (MB): {torch.cuda.max_memory_allocated() / (1024**2):.2f}")
    # Clear memory for next runs
    torch.cuda.empty_cache()

print("\nNote: The memory savings from these techniques are most pronounced with very large models and batch sizes.")


### Interpreting the Code and Performance Trade-offs

The provided code demonstrates the practical application of Mixed Precision Training and Gradient Checkpointing using PyTorch. While the memory savings won't be dramatically visible with our small example model, the structure illustrates how these techniques are integrated into a training loop.

#### Mixed Precision Training (`torch.cuda.amp`)

*   **`GradScaler()`**: This object manages the dynamic loss scaling. It keeps track of the scale factor, applies it to the loss before `backward()`, and then unscales the gradients before `optimizer.step()`. It also handles skipping optimizer steps if gradients become `NaN` or `Inf` due to numerical instability, adjusting the scale accordingly.
*   **`with autocast():`**: This context manager automatically casts operations within its scope to `float16` where appropriate, while keeping numerically sensitive operations in `float32`. This is the core of automatic mixed precision.

**Performance Implications:**
*   **Memory:** Significant reduction (often ~2x) in memory for activations and gradients, allowing larger models or batch sizes.
*   **Speed:** Often 1.5x to 3x speedup on modern GPUs with Tensor Cores (e.g., NVIDIA V100, A100, H100, RTX series). This is due to faster FP16 arithmetic and reduced memory bandwidth usage.
*   **Numerical Stability:** Generally robust with `GradScaler`, but requires careful monitoring. Very rarely, some custom operations might not be well-supported by `autocast` and could require manual casting or custom `autocast` policies.

#### Gradient Checkpointing (`torch.utils.checkpoint`)

*   **`checkpoint(function, *args, **kwargs)`**: This function takes a module or a function and its inputs. During the forward pass, it only stores the inputs to `function` and the output. During the backward pass, if gradients are needed for `function`, it re-runs `function` with the stored inputs to recompute the intermediate activations.
*   In our example, we apply `checkpoint` to individual `TransformerBlock` instances. In a real LLM, you might checkpoint entire groups of layers or even the entire model if memory is extremely constrained.

**Performance Implications:**
*   **Memory:** Drastic reduction in activation memory, often allowing models that would otherwise not fit into GPU memory to be trained. The memory footprint becomes roughly proportional to the number of *checkpointed segments* rather than the total number of layers.
*   **Speed:** Introduces a computational overhead because parts of the forward pass are recomputed during the backward pass. This can lead to a 10-30% slowdown in training speed, depending on the model architecture and checkpointing strategy.
*   **CPU Overhead:** The recomputation can sometimes lead to increased CPU usage if the recomputed parts are not fully GPU-bound.

#### Combining Mixed Precision and Gradient Checkpointing

As shown in the final section of the code, these two techniques are highly complementary and are almost always used together for training large LLMs.

*   Mixed precision reduces the memory footprint of the activations that *are* stored and speeds up the computations. It also reduces the memory footprint of the recomputed activations during checkpointing's backward pass.
*   Gradient checkpointing tackles the fundamental problem of storing *all* activations, allowing models with many layers to fit into memory.

By combining them, you get the best of both worlds: significantly reduced memory usage (from checkpointing) and faster training (from mixed precision), enabling the training of truly massive models that would be impossible otherwise.

**Typical Use Cases:**
*   **Training LLMs with billions of parameters:** Essential for fitting models onto available GPU memory.
*   **Increasing batch size:** If memory is freed up, you can often increase the batch size, which can lead to more stable gradients and faster convergence.
*   **Longer sequence lengths:** For models like Transformers, memory scales quadratically with sequence length. These techniques are crucial for handling very long contexts.

### Resources

*   **PyTorch Automatic Mixed Precision (AMP) Documentation:**
    *   [https://pytorch.org/docs/stable/amp.html](https://pytorch.org/docs/stable/amp.html)
*   **PyTorch Gradient Checkpointing Documentation:**
    *   [https://pytorch.org/docs/stable/checkpoint.html](https://pytorch.org/docs/stable/checkpoint.html)
*   **NVIDIA Blog Post on Mixed Precision Training:**
    *   [https://developer.nvidia.com/blog/accelerating-ai-training-with-mixed-precision/](https://developer.nvidia.com/blog/accelerating-ai-training-with-mixed-precision/)
*   **Hugging Face Accelerate Library (integrates AMP and Checkpointing seamlessly):**
    *   [https://huggingface.co/docs/accelerate/index](https://huggingface.co/docs/accelerate/index)
*   **Hugging Face Transformers Trainer (built-in support for AMP and Checkpointing):**
    *   [https://huggingface.co/docs/transformers/main_classes/trainer](https://huggingface.co/docs/transformers/main_classes/trainer)
